In [ ]:
# ============================================================
# SKETCHBYTE — CELL 3
# SCRIPT INPUT → QWEN3-4B → TTS-READY SCRIPT (STREAMING)
# ============================================================

import gc
import torch
import os
import re
import sys
from google.colab import output
from transformers import TextIteratorStreamer
from threading import Thread

# Enable custom widget manager fallback
output.enable_custom_widget_manager()

print("=" * 70)
print("SKETCHBYTE — CELL 3")
print("SCRIPT INPUT + TTS FORMATTING (STREAMING)")
print("=" * 70)

# ============================================================
# 1. CHECK QWEN3-4B
# ============================================================

print("\n[1/4] Checking Qwen3-4B formatter...")
print("-" * 70)

if "formatter_model" not in globals() or formatter_model is None:
    raise RuntimeError(
        "\n❌ Qwen3-4B formatter is not loaded.\n\n"
        "Run Cell 2 first."
    )

print("✅ Qwen3-4B ready.")

# ============================================================
# 2. SCRIPT INPUT (COLAB NATIVE FORM FALLBACK)
# ============================================================

print("\n[2/4] ENTER YOUR SKETCHBYTE SCRIPT")
print("-" * 70)

#@title Double-click to expand/collapse if form is hidden { display-mode: "form" }
#@markdown Enter or paste your script below, then run this cell to format it.
SCRIPT_INPUT_FORM = "He was, by any measure, one of the smartest men who ever lived. He reshaped our understanding of gravity, light, and motion itself. Universities still teach his laws four centuries later. And in the summer of 1720, this same man looked at his investment portfolio and made a decision that would cost him a fortune." #@param {type:"string"}

# ============================================================
# FORMATTER INSTRUCTION
# ============================================================

FORMATTER_INSTRUCTION = """
You are the professional TTS script preparation stage for SketchByte.
Your job is to prepare a YouTube narration for natural, human-sounding text-to-speech.

OUTPUT RULES (STRICT):
1. Output ONLY the narration text itself. No preamble, no intro, no outro, no commentary.
2. Do NOT rewrite the story. Do NOT change the meaning. Do NOT add new information.
3. Do NOT use markdown, bullet lists, headings, or divider lines.
4. Keep paragraphs separated by a single blank line.
5. Write numbers as words so they read naturally (e.g. seventeen twenty instead of 1720).
6. Expand abbreviations and symbols so a narrator can speak them (e.g. percent, and).
7. Keep punctuation that gives natural rhythm and pauses for speech.
"""

# ============================================================
# PROCESS SCRIPT
# ============================================================

raw_script = SCRIPT_INPUT_FORM.strip()

if not raw_script:
    print("\n❌ ERROR: Script input is empty. Please type or paste your script in the form field.")
else:
    word_count = len(raw_script.split())
    print(f"📝 Original word count: {word_count:,}")
    print("\n[3/4] Sending script to Qwen3-4B...")
    print("-" * 70)

    messages = [
        {"role": "system", "content": FORMATTER_INSTRUCTION},
        {"role": "user", "content": f"Prepare the following narration for natural TTS delivery:\n\n{raw_script}"}
    ]

    try:
        print("🧠 Qwen3-4B formatting progress (live stream below):\n")

        # Force return as dictionary to cleanly extract input_ids
        outputs = formatter_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            enable_thinking=False
        )

        input_ids = outputs["input_ids"].to(formatter_model.device)

        # Initialize the streamer
        streamer = TextIteratorStreamer(formatter_tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = dict(
            input_ids=input_ids,
            streamer=streamer,
            max_new_tokens=8192,
            do_sample=False,
            repetition_penalty=1.05
        )

        # Run generation in a separate thread so we can iterate over the streamer in the main thread
        thread = Thread(target=formatter_model.generate, kwargs=generation_kwargs)
        thread.start()

        # Retrieve stream and print live
        raw_formatted = ""
        for new_text in streamer:
            sys.stdout.write(new_text)
            sys.stdout.flush()
            raw_formatted += new_text

        thread.join()

        # Post-processing: remove any conversational framing the model may add
        cleaned_script = raw_formatted.strip()

        # If the model still used "---" dividers, keep only the narration parts.
        if "---" in cleaned_script:
            parts = cleaned_script.split("---")
            response_parts = [
                p for p in parts
                if p.strip() and not re.match(
                    r"^(Absolutely|Here's your|Sure,? here's?|Of course)",
                    p.strip(),
                    re.IGNORECASE,
                )
            ]
            cleaned_script = "\n".join(response_parts).strip()

        # Drop any "Sure, here is..." type framing before the real narration.
        cleaned_script = re.sub(
            r"^(Sure|Absolutely|Of course|Here's|Below is|Here is).*?\n{2,}",
            "",
            cleaned_script,
            flags=re.IGNORECASE | re.DOTALL,
        )

        # Drop any trailing commentary after the narration ends.
        cleaned_script = re.sub(
            r"\n{2,}(This version|Perfect for YouTube|Hope this helps|Let me know|Feel free).*?$",
            "",
            cleaned_script,
            flags=re.IGNORECASE | re.DOTALL,
        )

        formatted_script = cleaned_script.strip()
        formatted_words = len(formatted_script.split())

        print("\n\n" + "=" * 70)
        print("✅ FORMATTING & CLEANING COMPLETE")
        print("=" * 70)
        print(f"\nOriginal words : {word_count:,}")
        print(f"Formatted words: {formatted_words:,}")

        output_file = "/content/SketchByte_TTS_Ready_Script.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(formatted_script)

        print(f"\n💾 SCRIPT SAVED to: {output_file}")
        print("\n🟢 CELL 3 COMPLETE. Proceed to Cell 4!")

    except Exception as e:
        print(f"\n❌ FORMATTING FAILED: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise